In [18]:
# CHUNK 1: Import Libraries and Initialize Variables

# Standard libraries for file and folder operations
import os
import io
import glob

# Third-party libraries for data manipulation, web requests, and PDF parsing
import pandas as pd     
import requests         
import pdfplumber       

# ---------------------------------------------------------
# PATH INITIALIZATION
# ---------------------------------------------------------

# 1. Direct path to your main project folder
ROOT_DIR = r"C:\Users\Samarth Chaudhary\Downloads\RIDHI SIDHI COMMERCIAL HUB BLOCK A"

# 2. Construct the deep path to where the QPR folders actually live
# This bridges the gap: ROOT_DIR -> sub -> DREAM COLONY
QPR_BASE_PATH = os.path.join(ROOT_DIR, "sub", "RIDHI SIDHI COMMERCIAL HUB BLOCK A")

# ---------------------------------------------------------
# VERIFICATION
# ---------------------------------------------------------
print("--- Initializing Chunk 1 ---")

if os.path.exists(QPR_BASE_PATH):
    print(f"✅ Success: Base path locked in at:\n   {QPR_BASE_PATH}")
    
    # Find and list all quarterly subfolders inside it
    qpr_folders = [f for f in os.listdir(QPR_BASE_PATH) if os.path.isdir(os.path.join(QPR_BASE_PATH, f))]
    print(f"📁 Found {len(qpr_folders)} QPR folders ready for processing.")
    
    # Print the first few folders to confirm it's reading correctly
    if qpr_folders:
        print("Preview of folders found:")
        for folder in qpr_folders[:3]: # Shows up to 3 folders as a quick check
            print(f"   - {folder}")
else:
    print(f"❌ Error: Path not found! Please check if the folder exists at:\n   {QPR_BASE_PATH}")

print("----------------------------")

--- Initializing Chunk 1 ---
✅ Success: Base path locked in at:
   C:\Users\Samarth Chaudhary\Downloads\RIDHI SIDHI COMMERCIAL HUB BLOCK A\sub\RIDHI SIDHI COMMERCIAL HUB BLOCK A
📁 Found 4 QPR folders ready for processing.
Preview of folders found:
   - April to June, 2025
   - January to March, 2026
   - July to September, 2025
----------------------------


In [8]:
import os
import io
import re
import pandas as pd
import requests
from pdf2image import convert_from_bytes
import easyocr

# =========================================================================
# 1. PATH CONFIGURATION
# =========================================================================
ROOT_DIR = r"C:\Users\Samarth Chaudhary\Downloads\RIDHI SIDHI COMMERCIAL HUB BLOCK A"
QPR_BASE_PATH = os.path.join(ROOT_DIR, "sub", "RIDHI SIDHI COMMERCIAL HUB BLOCK A")

POPPLER_PATH = None
poppler_base = r"C:\Users\Samarth Chaudhary\Downloads\poppler"

print("--- Locating Poppler Dependency ---")
if os.path.exists(poppler_base):
    for root, dirs, files in os.walk(poppler_base):
        if "pdftoppm.exe" in files:
            POPPLER_PATH = root
            break

if POPPLER_PATH:
    print(f"✅ Success: Linked Poppler engine binaries at:\n   {POPPLER_PATH}\n")
else:
    print(f"❌ Warning: Could not find 'pdftoppm.exe' inside {poppler_base}\n")

# =========================================================================
# 2. ADVANCED DATA NORMALIZATION ENGINE (HANDWRITING + TYPED HYBRID)
# =========================================================================
print("--- Initializing Adaptive OCR Engine ---")
reader = easyocr.Reader(['en'], gpu=False) 

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

def clean_handwritten_proposed(val_str):
    """Normalizes both typed text and distorted cursive handwriting to Yes/No."""
    if isinstance(val_str, pd.Series):
        val_str = val_str.iloc[0]
    if not val_str or pd.isna(val_str):
        return "No"
        
    clean_str = str(val_str).strip().lower()
    
    # Comprehensive cursive handwriting mapping table for the word "yes"
    cursive_yes_signals = [
        'yes', 'ycs', 'ves', 'vcs', 'ys', 've', 'ye', 'y5', '4es', '7es',
        'ues', 'vces', 'uces', 'gcs', 'y_s', 'v.s', 'vis', 'g3s', 'les'
    ]
    
    if any(signal in clean_str for signal in cursive_yes_signals):
        return "Yes"
    return "No"

def clean_handwritten_percentage(val_str, proposed_status):
    """Cleans up percentage text while filtering out handwritten stroke noise."""
    if isinstance(val_str, pd.Series):
        val_str = val_str.iloc[0]
    if pd.isna(val_str):
        return 0
        
    # FIX: If the value is already a native number from Excel, process it safely directly
    if isinstance(val_str, (int, float)):
        return int(round(val_str))
        
    clean_str = str(val_str).strip().lower()
    
    # Direct check for handwritten N/A variations
    if any(x in clean_str for x in ['na', 'n/a', 'ni', 'la', '7a', 'none', 'blan', 'v']):
        return 0
        
    # Standardize common character-to-number transitions
    clean_str = clean_str.replace('o', '0').replace('l', '1').replace('s', '5').replace('i', '1').replace('q', '9')
    
    digits = re.sub(r'\D', '', clean_str)
    if not digits:
        return 0
        
    num = int(digits)
    
    # Defense against cursive % tail strokes being read as trailing digits
    str_num = str(num)
    if num > 100:
        if any(str_num.endswith(ch) for ch in ['1', '4', '6', '9', '0']):
            possible_perc = int(str_num[:-1])
            if possible_perc <= 100:
                return possible_perc
        if str_num.startswith('100'):
            return 100
        if str_num.startswith('90'):
            return 90
            
    if num > 100:
        return 0
        
    return num

def find_handwritten_row_bounds(s_no, task_name, ocr_results, page_width):
    """Finds the precise top and bottom boundaries of a row dynamically."""
    target_sno = str(s_no).strip().rstrip('.')
    keywords = [w.lower() for w in re.split(r'\W+', str(task_name)) if len(w) > 2]
    
    best_y_top = None
    best_y_bottom = None
    max_score = 0
    
    for box, text, conf in ocr_results:
        text_clean = text.strip().lower().rstrip('.')
        x_min = min(pt[0] for pt in box)
        y_min = min(pt[1] for pt in box)
        y_max = max(pt[1] for pt in box)
        
        score = 0
        if x_min < page_width * 0.25:
            if text_clean == target_sno or text_clean.startswith(target_sno + " ") or text_clean.startswith(target_sno + "."):
                score += 60 
        
        word_matches = sum(2 for kw in keywords if kw in text_clean)
        score += word_matches
        
        if score > max_score and score >= 2:
            max_score = score
            best_y_top = y_min
            best_y_bottom = y_max
            
    return best_y_top, best_y_bottom

def extract_columns_by_vertical_overlap(y_top, y_bottom, ocr_results, page_width):
    """Uses relative area intersection over vertical spans to find matching cells."""
    if y_top is None or y_bottom is None:
        return "No", 0
        
    row_height = y_bottom - y_top
    right_side_elements = []
    
    for box, text, conf in ocr_results:
        box_y_min = min(pt[1] for pt in box)
        box_y_max = max(pt[1] for pt in box)
        box_x_center = sum(pt[0] for pt in box) / 4.0
        
        overlap_len = min(y_bottom, box_y_max) - max(y_top, box_y_min)
        
        if overlap_len > 0:
            box_height = box_y_max - box_y_min
            overlap_ratio = overlap_len / max(1, box_height)
            
            if overlap_ratio > 0.25 or overlap_len > (row_height * 0.25):
                if box_x_center > page_width * 0.45: 
                    right_side_elements.append((box_x_center, text.strip()))
                    
    if not right_side_elements:
        return "No", 0
        
    right_side_elements.sort(key=lambda item: item[0])
    
    proposed_raw = ""
    percentage_raw = ""
    
    for x_center, text_val in right_side_elements:
        if x_center < page_width * 0.72:
            proposed_raw += " " + text_val
        else:
            percentage_raw += " " + text_val
            
    if not percentage_raw.strip() and len(right_side_elements) >= 2:
        proposed_raw = right_side_elements[0][1]
        percentage_raw = right_side_elements[-1][1]
    elif not percentage_raw.strip() and len(right_side_elements) == 1:
        combined_text = right_side_elements[0][1]
        digits = re.sub(r'\D', '', combined_text)
        if digits:
            percentage_raw = digits
            proposed_raw = combined_text
        else:
            proposed_raw = combined_text
            percentage_raw = "0"

    final_proposed = clean_handwritten_proposed(proposed_raw)
    final_percentage = clean_handwritten_percentage(percentage_raw, final_proposed)
    
    return final_proposed, final_percentage


# =========================================================================
# 3. CORE PROCESSING ENGINE 
# =========================================================================
def main():
    if not os.path.exists(QPR_BASE_PATH):
        print(f"❌ Error: Cannot find path: {QPR_BASE_PATH}")
        return

    qpr_folders = [f for f in os.listdir(QPR_BASE_PATH) if os.path.isdir(os.path.join(QPR_BASE_PATH, f))]
    print(f"📂 Found {len(qpr_folders)} quarterly target directories.")

    for folder in qpr_folders:
        folder_path = os.path.join(QPR_BASE_PATH, folder)
        
        # Guard added to prevent evaluating an already generated output file
        possible_files = [
            f for f in os.listdir(folder_path) 
            if f.startswith("Form_R1") and not f.startswith("~$") and "compared" not in f.lower()
        ]
        valid_files = [f for f in possible_files if f.lower().endswith(('.xlsx', '.xls', '.csv'))]
        
        if not valid_files:
            continue
            
        r1_filename = valid_files[0]
        r1_file_path = os.path.join(folder_path, r1_filename)
            
        print(f"\n🚀 Evaluating Directory: {folder} | Target: {r1_filename}")
        
        if r1_file_path.lower().endswith('.csv'):
            df = pd.read_csv(r1_file_path)
        else:
            df = pd.read_excel(r1_file_path)
        
        if 'docs' not in df.columns or 'S.No.' not in df.columns:
            print("   ❌ Error: Column 'docs' or 'S.No.' is missing.")
            continue
            
        pdf_url = df['docs'].iloc[0]
        if pd.isna(pdf_url) or not str(pdf_url).startswith(('http://', 'https://')):
            print("   ❌ Error: Document source link is invalid.")
            continue
            
        print(f"   📥 Fetching PDF streams...")
        try:
            response = requests.get(pdf_url, headers=HEADERS, timeout=30)
            response.raise_for_status()
            
            pages = convert_from_bytes(response.content, dpi=300, poppler_path=POPPLER_PATH)
            
            best_page_ocr = []
            best_page_score = -1
            page_width = 1000
            
            table_b_keywords = ["TABLE-B", "TABLE B", "INTERNAL ROADS", "WATER SUPPLY", "STREET LIGHTING", "STORMS WATER", "AMENITIES"]
            
            for i, page_img in enumerate(pages):
                w, h = page_img.size
                img_byte_arr = io.BytesIO()
                page_img.save(img_byte_arr, format='PNG')
                
                page_ocr = reader.readtext(img_byte_arr.getvalue(), detail=1)
                
                page_text = " ".join([b[1] for b in page_ocr]).upper()
                score = sum(1 for kw in table_b_keywords if kw in page_text)
                
                if score > best_page_score:
                    best_page_score = score
                    best_page_ocr = page_ocr
                    page_width = w
                    if score > 2:
                        print(f"     Found target sheet matching metadata on Page {i+1} (Score: {score})")
            
            if not best_page_ocr or best_page_score == 0:
                print("   ❌ Error: Structural layout matching Table-B could not be verified.")
                continue
                
            extracted_proposed_cols = []
            extracted_percentage_cols = []
            comparison_results = []
            
            # FIX: Find target Excel column using a fuzzy, prioritized search strategy
            current_q_col = [c for c in df.columns if 'percentage' in str(c).lower() and 'current' in str(c).lower()]
            last_q_col = [c for c in df.columns if 'percentage' in str(c).lower() and 'last' in str(c).lower()]
            
            chosen_col = None
            if current_q_col:
                chosen_col = current_q_col[0]
            elif last_q_col:
                chosen_col = last_q_col[0]
                
            if chosen_col:
                print(f"   🎯 Matching against Excel column: '{chosen_col}'")
            else:
                print("   ⚠️  Warning: No matching percentage column found in Excel. Defaulting target to 0.")

            for index, row in df.iterrows():
                s_no = row['S.No.']
                task = row['Task/Facility']
                
                y_top_bound, y_bottom_bound = find_handwritten_row_bounds(s_no, task, best_page_ocr, page_width)
                pdf_prop, pdf_perc = extract_columns_by_vertical_overlap(y_top_bound, y_bottom_bound, best_page_ocr, page_width)
                
                extracted_proposed_cols.append(pdf_prop)
                extracted_percentage_cols.append(pdf_perc)
                
                # Extract clean target value using the prioritized column
                target_val = row[chosen_col] if chosen_col else 0
                excel_perc_clean = clean_handwritten_percentage(target_val, "Yes")
                
                status = "Match" if excel_perc_clean == pdf_perc else "Mismatch"
                comparison_results.append(status)
                
                print(f"     Row {s_no} -> PDF Found: [{pdf_prop} | {pdf_perc}%] vs Target: [{excel_perc_clean}%] -> {status}")
                    
            # Safe clean drop of target structural tracking columns before write execution
            df = df.drop(columns=['PDF_Proposed', 'PDF_Percentage', 'Comparison_Status'], errors='ignore')
                    
            docs_position = df.columns.get_loc('docs')
            df.insert(docs_position, 'PDF_Proposed', extracted_proposed_cols)
            df.insert(docs_position + 1, 'PDF_Percentage', extracted_percentage_cols)
            df.insert(docs_position + 2, 'Comparison_Status', comparison_results)
            
            if r1_file_path.lower().endswith('.csv'):
                output_path = os.path.join(folder_path, "Form_R1_Compared.csv")
                df.to_csv(output_path, index=False)
            else:
                output_path = os.path.join(folder_path, "Form_R1_Compared.xlsx")
                df.to_excel(output_path, index=False)
                
            print(f"   ✅ Extraction and array validation written successfully: {os.path.basename(output_path)}")
            
        except Exception as e:
            print(f"   ❌ Execution halted for this folder: {e}")

if __name__ == "__main__":
    main()
    print("\n=================== Check Execution Finished ===================")

Using CPU. Note: This module is much faster with a GPU.


--- Locating Poppler Dependency ---
✅ Success: Linked Poppler engine binaries at:
   C:\Users\Samarth Chaudhary\Downloads\poppler\Poopler\Library\bin

--- Initializing Adaptive OCR Engine ---
📂 Found 4 quarterly target directories.

🚀 Evaluating Directory: April to June, 2025 | Target: Form_R1.xlsx
   📥 Fetching PDF streams...


c:\Users\Samarth Chaudhary\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


     Found target sheet matching metadata on Page 3 (Score: 5)
   🎯 Matching against Excel column: 'Percentage (Current Quarter)'
     Row 1 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
     Row 2 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
     Row 3 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
     Row 4 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
     Row 5 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
     Row 6 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
     Row 7 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
     Row 8 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
     Row 9 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
     Row 10 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
     Row 11 -> PDF Found: [No | 0%] vs Target: [0%] -> Match
     Row 12 -> PDF Found: [No | 0%] vs Target: [0%] -> Match
     Row 13 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
   ✅ Extraction and array validation written successf

c:\Users\Samarth Chaudhary\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


     Found target sheet matching metadata on Page 3 (Score: 5)
   🎯 Matching against Excel column: 'Percentage (Current Quarter)'
     Row 1 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
     Row 2 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
     Row 3 -> PDF Found: [No | 0%] vs Target: [0%] -> Match
     Row 4 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
     Row 5 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
     Row 6 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
     Row 7 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
     Row 8 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
     Row 9 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
     Row 10 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
     Row 11 -> PDF Found: [No | 0%] vs Target: [0%] -> Match
     Row 12 -> PDF Found: [No | 0%] vs Target: [0%] -> Match
     Row 13 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
   ✅ Extraction and array validation written successfull

c:\Users\Samarth Chaudhary\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


     Found target sheet matching metadata on Page 3 (Score: 5)
   🎯 Matching against Excel column: 'Percentage (Current Quarter)'
     Row 1 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
     Row 2 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
     Row 3 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
     Row 4 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
     Row 5 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
     Row 6 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
     Row 7 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
     Row 8 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
     Row 9 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
     Row 10 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
     Row 11 -> PDF Found: [No | 0%] vs Target: [0%] -> Match
     Row 12 -> PDF Found: [No | 0%] vs Target: [0%] -> Match
     Row 13 -> PDF Found: [No | 5%] vs Target: [0%] -> Mismatch
   ✅ Extraction and array validation written successf

In [4]:
import os

# Paste your exact path inside the quotes below
ROOT_DIR =   r"C:\Users\Samarth Chaudhary\Downloads\SHIV ENCLAVE"

print(f"Checking folder: {ROOT_DIR}")

if os.path.exists(ROOT_DIR):
    print("✅ Folder found! Looking for Excel files...\n")
    found_any = False
    for root, dirs, files in os.walk(ROOT_DIR):
        for file in files:
            if file.startswith("Form_R2"):
                print(f"Found target file: {os.path.join(root, file)}")
                found_any = True
    
    if not found_any:
        print("❌ Folder exists, but couldn't find any files starting with 'Form_R2'.")
else:
    print("❌ Folder DOES NOT exist. The path is wrong.")

Checking folder: C:\Users\Samarth Chaudhary\Downloads\SHIV ENCLAVE
✅ Folder found! Looking for Excel files...

Found target file: C:\Users\Samarth Chaudhary\Downloads\SHIV ENCLAVE\sub\Shiv Enclave\April to June, 2020\Form_R2.xlsx
Found target file: C:\Users\Samarth Chaudhary\Downloads\SHIV ENCLAVE\sub\Shiv Enclave\April to June, 2021\Form_R2.xlsx
Found target file: C:\Users\Samarth Chaudhary\Downloads\SHIV ENCLAVE\sub\Shiv Enclave\April to June, 2021\Form_R2_Compared.xlsx
Found target file: C:\Users\Samarth Chaudhary\Downloads\SHIV ENCLAVE\sub\Shiv Enclave\April to June, 2022\Form_R2.xlsx
Found target file: C:\Users\Samarth Chaudhary\Downloads\SHIV ENCLAVE\sub\Shiv Enclave\April to June, 2022\Form_R2_Compared.xlsx
Found target file: C:\Users\Samarth Chaudhary\Downloads\SHIV ENCLAVE\sub\Shiv Enclave\April to June, 2023\Form_R2.xlsx
Found target file: C:\Users\Samarth Chaudhary\Downloads\SHIV ENCLAVE\sub\Shiv Enclave\April to June, 2023\Form_R2_Compared.xlsx
Found target file: C:\Users\S

In [5]:
import os
import io
import re
import pandas as pd
import requests
from pdf2image import convert_from_bytes
import easyocr

# =========================================================================
# 1. PATH CONFIGURATION
# =========================================================================
ROOT_DIR = r"C:\Users\Samarth Chaudhary\Downloads\SHIV ENCLAVE"
QPR_BASE_PATH = os.path.join(ROOT_DIR, "sub", "SHIV ENCLAVE")

POPPLER_PATH = None
poppler_base = r"C:\Users\Samarth Chaudhary\Downloads\poppler"

print("--- Locating Poppler Dependency ---")
if os.path.exists(poppler_base):
    for root, dirs, files in os.walk(poppler_base):
        if "pdftoppm.exe" in files:
            POPPLER_PATH = root
            break

if POPPLER_PATH:
    print(f"✅ Poppler found at:\n   {POPPLER_PATH}\n")
else:
    print(f"❌ Warning: Could not find 'pdftoppm.exe' inside {poppler_base}\n")

# =========================================================================
# 2. OCR ENGINE INIT
# =========================================================================
print("--- Initializing Adaptive OCR Engine ---")
reader = easyocr.Reader(['en'], gpu=False)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

# =========================================================================
# 3. CLEANING FUNCTIONS
# =========================================================================

def clean_amount(val_str):
    """
    Cleans any amount value — from OCR or Excel.
    - Strips Rs., %, commas, dots used as thousand separators
    - Treats blank / NA / N/A / None as 0
    - Returns plain numeric string e.g. '1980000', '66', '0'
    - Does NOT include Rs. or % in output
    """
    if val_str is None:
        return "0"
    if isinstance(val_str, float) and pd.isna(val_str):
        return "0"

    raw = str(val_str).strip().lower()

    # Treat blanks and NA variants as 0
    if raw in ('', 'na', 'n/a', 'none', 'nil', '-', 'nan', 'blank'):
        return "0"

    # Strip currency and percent markers (never printed in output)
    raw = re.sub(r'[Rr][Ss]\.?\s*', '', raw)
    raw = re.sub(r'%', '', raw)

    # OCR noise: common character substitutions
    raw = raw.replace('o', '0').replace('l', '1').replace('i', '1') \
             .replace('s', '5').replace('q', '9').replace('b', '6')

    # Remove all non-numeric characters except minus
    raw = re.sub(r'[^\d\-]', '', raw)

    if not raw or raw == '-':
        return "0"

    # Guard against runaway OCR reads
    if len(raw) > 12:
        raw = raw[:12]

    # Remove leading zeros but keep single "0"
    raw = raw.lstrip('0') or "0"

    return raw


def normalize_excel_amount(val_str):
    """
    Same cleaner but handles formatted Excel numbers like
    '30,00,000' or '1980000.0' before comparing with OCR output.
    """
    if val_str is None:
        return "0"
    if isinstance(val_str, (int, float)):
        if pd.isna(val_str):
            return "0"
        # Remove trailing .0 from floats
        raw = str(int(round(float(val_str))))
        raw = raw.lstrip('0') or "0"
        return raw

    raw = str(val_str).strip().lower()
    if raw in ('', 'na', 'n/a', 'none', 'nil', '-', 'nan', 'blank'):
        return "0"

    raw = re.sub(r'[Rr][Ss]\.?\s*', '', raw)
    raw = re.sub(r'%', '', raw)
    raw = re.sub(r'[^\d\-]', '', raw)

    if not raw or raw == '-':
        return "0"

    raw = raw.lstrip('0') or "0"
    return raw


# =========================================================================
# 4. ROW BOUNDARY DETECTION
# =========================================================================

def find_row_bounds(s_no, particulars, ocr_results, page_width):
    """
    Finds the vertical Y band for a given row in the OCR result set
    by matching the serial number and keywords from the 'Particulars' column.
    """
    target_sno = str(s_no).strip().rstrip('.')
    keywords = [w.lower() for w in re.split(r'\W+', str(particulars)) if len(w) > 2]

    best_y_top = None
    best_y_bottom = None
    max_score = 0

    for box, text, conf in ocr_results:
        text_clean = text.strip().lower().rstrip('.')
        x_min = min(pt[0] for pt in box)
        y_min = min(pt[1] for pt in box)
        y_max = max(pt[1] for pt in box)

        score = 0
        # S.No must appear in the left margin
        if x_min < page_width * 0.25:
            if (text_clean == target_sno
                    or text_clean.startswith(target_sno + " ")
                    or text_clean.startswith(target_sno + ".")):
                score += 60

        score += sum(2 for kw in keywords if kw in text_clean)

        if score > max_score and score >= 2:
            max_score = score
            best_y_top = y_min
            best_y_bottom = y_max

    return best_y_top, best_y_bottom


# =========================================================================
# 5. AMOUNT COLUMN EXTRACTOR (Column 3 — rightmost column)
# =========================================================================

def extract_amount_from_row(y_top, y_bottom, ocr_results, page_width):
    """
    Given the vertical bounds of a row, collects all OCR tokens
    that fall in the rightmost column zone (x > 55% of page width)
    and returns a cleaned numeric string.
    Handles both typed and handwritten values.
    """
    if y_top is None or y_bottom is None:
        return "0"

    row_height = max(1, y_bottom - y_top)
    candidates = []

    for box, text, conf in ocr_results:
        box_y_min = min(pt[1] for pt in box)
        box_y_max = max(pt[1] for pt in box)
        box_x_center = sum(pt[0] for pt in box) / 4.0

        overlap = min(y_bottom, box_y_max) - max(y_top, box_y_min)
        if overlap <= 0:
            continue

        box_height = max(1, box_y_max - box_y_min)
        overlap_ratio = overlap / box_height

        # Accept token if it overlaps meaningfully with the row band
        if overlap_ratio > 0.25 or overlap > row_height * 0.25:
            # Only look at the right-side column (Amounts col 3)
            if box_x_center > page_width * 0.55:
                candidates.append((box_x_center, text.strip()))

    if not candidates:
        return "0"

    # Sort left-to-right and concatenate all fragments in the zone
    candidates.sort(key=lambda item: item[0])
    raw_combined = " ".join(t for _, t in candidates)

    return clean_amount(raw_combined)


# =========================================================================
# 6. FIND THE BEST TABLE-B PAGE IN THE PDF
# =========================================================================

TABLE_B_KEYWORDS = [
    "TABLE B", "TABLE-B",
    "TOTAL ESTIMATED COST",
    "COST INCURRED",
    "WORK DONE IN PERCENTAGE",
    "BALANCE COST",
    "ADDITIONAL",
    "DEVELOPMENT WORKS",
    "REAL ESTATE PROJECT"
]

def find_table_b_page(pages):
    """
    Scans all pages and returns the OCR result set + page width
    for the page most likely to contain Table B.
    """
    best_ocr = []
    best_score = -1
    best_width = 1000

    for i, page_img in enumerate(pages):
        w, h = page_img.size
        img_byte_arr = io.BytesIO()
        page_img.save(img_byte_arr, format='PNG')

        page_ocr = reader.readtext(img_byte_arr.getvalue(), detail=1)
        page_text = " ".join([b[1] for b in page_ocr]).upper()

        score = sum(1 for kw in TABLE_B_KEYWORDS if kw in page_text)

        if score > best_score:
            best_score = score
            best_ocr = page_ocr
            best_width = w
            if score > 2:
                print(f"     📄 Table B candidate: Page {i+1} (Score: {score})")

    return best_ocr, best_score, best_width


# =========================================================================
# 7. CORE PROCESSING ENGINE
# =========================================================================

def main():
    if not os.path.exists(QPR_BASE_PATH):
        print(f"❌ Error: Cannot find path: {QPR_BASE_PATH}")
        return

    qpr_folders = [
        f for f in os.listdir(QPR_BASE_PATH)
        if os.path.isdir(os.path.join(QPR_BASE_PATH, f))
    ]
    print(f"📂 Found {len(qpr_folders)} quarterly directories.\n")

    for folder in sorted(qpr_folders):
        folder_path = os.path.join(QPR_BASE_PATH, folder)

        # ── Find Form_R2 file ──
        possible = [
            f for f in os.listdir(folder_path)
            if f.startswith("Form_R2") and not f.startswith("~$")
        ]
        valid = [f for f in possible if f.lower().endswith(('.xlsx', '.xls', '.csv'))]

        if not valid:
            print(f"⚠️  No Form_R2 file in: {folder} — skipping.")
            continue

        r2_filename = valid[0]
        r2_file_path = os.path.join(folder_path, r2_filename)
        print(f"\n🚀 Processing: {folder}  |  File: {r2_filename}")

        # ── Load Excel ──
        if r2_file_path.lower().endswith('.csv'):
            df = pd.read_csv(r2_file_path)
        else:
            df = pd.read_excel(r2_file_path)

        # ── Validate required columns ──
        if 'docs' not in df.columns or 'S.No.' not in df.columns:
            print("   ❌ Missing 'docs' or 'S.No.' column — skipping.")
            continue

        pdf_url = df['docs'].iloc[0]
        if pd.isna(pdf_url) or not str(pdf_url).startswith(('http://', 'https://')):
            print("   ❌ Invalid PDF URL — skipping.")
            continue

        # ── Fetch & render PDF ──
        print("   📥 Downloading PDF...")
        try:
            response = requests.get(pdf_url, headers=HEADERS, timeout=30)
            response.raise_for_status()
            pages = convert_from_bytes(response.content, dpi=300, poppler_path=POPPLER_PATH)
            print(f"   🖼️  Rendered {len(pages)} page(s) at 300 DPI.")
        except Exception as e:
            print(f"   ❌ Failed to fetch/render PDF: {e}")
            continue

        # ── Find Table B page ──
        best_ocr, best_score, page_width = find_table_b_page(pages)

        if not best_ocr or best_score == 0:
            print("   ❌ Table B not detected in PDF — skipping.")
            continue

        # ── Decide which Excel column to compare against ──
        if 'By the end of Current Quarter' in df.columns:
            compare_col = 'By the end of Current Quarter'
        elif 'By the end of Last Quarter' in df.columns:
            compare_col = 'By the end of Last Quarter'
        else:
            compare_col = None
            print("   ⚠️  No comparison column found — Comparison_Status will be 'N/A'.")

        # ── Row label column ──
        label_col = 'Particulars' if 'Particulars' in df.columns else df.columns[1]

        # ── Process each row ──
        pdf_amounts = []
        comparison_statuses = []

        for idx, row in df.iterrows():
            s_no       = row['S.No.']
            particulars = row[label_col]

            # Find row on PDF page
            y_top, y_bottom = find_row_bounds(s_no, particulars, best_ocr, page_width)

            # Extract amount from rightmost column
            pdf_val = extract_amount_from_row(y_top, y_bottom, best_ocr, page_width)
            pdf_amounts.append(pdf_val)

            # Compare with Excel value
            if compare_col:
                excel_val = normalize_excel_amount(row[compare_col])
                status = "Match" if excel_val == pdf_val else "Mismatch"
            else:
                status = "N/A"

            comparison_statuses.append(status)
            print(f"     Row {s_no:>3} | PDF: [{pdf_val:>12}] | Excel: [{normalize_excel_amount(row[compare_col]) if compare_col else 'N/A':>12}] | {status}")

        # ── Insert new columns just before 'docs' ──
        for col in ['PDF_Amount', 'Comparison_Status']:
            if col in df.columns:
                df.drop(columns=[col], inplace=True)

        insert_at = df.columns.get_loc('docs')
        df.insert(insert_at,     'PDF_Amount',         pdf_amounts)
        df.insert(insert_at + 1, 'Comparison_Status',  comparison_statuses)

        # ── Save output file ──
        if r2_file_path.lower().endswith('.csv'):
            out_path = os.path.join(folder_path, "Form_R2_Compared.csv")
            df.to_csv(out_path, index=False)
        else:
            out_path = os.path.join(folder_path, "Form_R2_Compared.xlsx")
            df.to_excel(out_path, index=False)

        print(f"   ✅ Saved: {os.path.basename(out_path)}")

    print("\n=================== R2 Comparison Finished ===================")


if __name__ == "__main__":
    main()

Using CPU. Note: This module is much faster with a GPU.


--- Locating Poppler Dependency ---
✅ Poppler found at:
   C:\Users\Samarth Chaudhary\Downloads\poppler\Poopler\Library\bin

--- Initializing Adaptive OCR Engine ---
📂 Found 25 quarterly directories.


🚀 Processing: April to June, 2020  |  File: Form_R2.xlsx
   ❌ Invalid PDF URL — skipping.

🚀 Processing: April to June, 2021  |  File: Form_R2.xlsx
   📥 Downloading PDF...
   🖼️  Rendered 4 page(s) at 300 DPI.


c:\Users\Samarth Chaudhary\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


     📄 Table B candidate: Page 2 (Score: 7)
     📄 Table B candidate: Page 3 (Score: 8)
     Row   1 | PDF: [  1015080061] | Excel: [  3777300000] | Mismatch
     Row   2 | PDF: [ 51151116011] | Excel: [  1550974500] | Mismatch
     Row   3 | PDF: [          41] | Excel: [        4100] | Mismatch
     Row   4 | PDF: [650510522263] | Excel: [  2226325500] | Mismatch
     Row   5 | PDF: [       11151] | Excel: [           0] | Mismatch
   ✅ Saved: Form_R2_Compared.xlsx

🚀 Processing: April to June, 2022  |  File: Form_R2.xlsx
   📥 Downloading PDF...
   🖼️  Rendered 3 page(s) at 300 DPI.


c:\Users\Samarth Chaudhary\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...

     📄 Table B candidate: Page 2 (Score: 7)
     📄 Table B candidate: Page 3 (Score: 8)
     Row   1 | PDF: [           0] | Excel: [  3777300000] | Mismatch
     Row   2 | PDF: [ 51151116011] | Excel: [  2020396000] | Mismatch
     Row   3 | PDF: [           0] | Excel: [        5300] | Mismatch
     Row   4 | PDF: [        4000] | Excel: [  1756904000] | Mismatch
     Row   5 | PDF: [           -] | Excel: [           0] | Mismatch
   ✅ Saved: Form_R2_Compared.xlsx

🚀 Processing: October to December, 2024  |  File: Form_R2.xlsx
   📥 Downloading PDF...
   🖼️  Rendered 3 page(s) at 300 DPI.


c:\Users\Samarth Chaudhary\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


     📄 Table B candidate: Page 1 (Score: 3)
     📄 Table B candidate: Page 2 (Score: 6)
     📄 Table B candidate: Page 3 (Score: 7)
     Row   1 | PDF: [  1500111505] | Excel: [  3777300000] | Mismatch
     Row   2 | PDF: [        4710] | Excel: [  1779675100] | Mismatch
     Row   3 | PDF: [           0] | Excel: [        4700] | Mismatch
     Row   4 | PDF: [           0] | Excel: [  1997624900] | Mismatch
     Row   5 | PDF: [           0] | Excel: [           0] | Match
   ✅ Saved: Form_R2_Compared.xlsx

🚀 Processing: October to December, 2025  |  File: Form_R2.xlsx
   📥 Downloading PDF...
   🖼️  Rendered 4 page(s) at 300 DPI.


c:\Users\Samarth Chaudhary\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


     📄 Table B candidate: Page 1 (Score: 3)
     📄 Table B candidate: Page 2 (Score: 6)
     📄 Table B candidate: Page 3 (Score: 8)
     Row   1 | PDF: [  3777300000] | Excel: [  3777300000] | Match
     Row   2 | PDF: [  2020396000] | Excel: [  2020396000] | Match
     Row   3 | PDF: [          53] | Excel: [        5300] | Mismatch
     Row   4 | PDF: [           0] | Excel: [  1756904000] | Mismatch
     Row   5 | PDF: [  3777300000] | Excel: [           0] | Mismatch
   ✅ Saved: Form_R2_Compared.xlsx

=================== R2 Comparison Finished ===================


In [23]:
import os
import pandas as pd
import requests
import re
import tempfile
from pdf2image import convert_from_path
import pytesseract
import cv2
import numpy as np

# ==============================================================================
# CONFIGURATION PATHS
ROOT_DIR = r"C:\Users\Samarth Chaudhary\Downloads\RIDHI SIDHI COMMERCIAL HUB BLOCK A"
POPPLER_PATH = r"C:\Users\Samarth Chaudhary\Downloads\poppler\Poopler\Library\bin"
TESSERACT_PATH = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
# ==============================================================================

pytesseract.pytesseract.tesseract_cmd = TESSERACT_PATH

def download_pdf(url):
    """Downloads a PDF from a given URL to a temporary file safely."""
    try:
        response = requests.get(url, stream=True, timeout=30)
        response.raise_for_status()
        temp_pdf = tempfile.NamedTemporaryFile(delete=False, suffix=".pdf")
        with open(temp_pdf.name, 'wb') as f:
            f.write(response.content)
        return temp_pdf.name
    except Exception as e:
        print(f"❌ Error downloading PDF from {url}: {e}")
        return None

def clean_amount(val):
    """Converts Excel cell values into clean, standardized floats for comparison."""
    if pd.isna(val):
        return 0.0
    val_str = str(val).strip().upper()
    if val_str in ["", "NA", "N/A", "NONE", "NULL", "0"]:
        return 0.0
    val_str = val_str.replace("RS.", "").replace("RS", "").replace(",", "").strip()
    try:
        return float(val_str)
    except ValueError:
        return 0.0

def extract_r3_top_amounts_from_pdf(pdf_path):
    """
    Extracts the Estimated and Incurred amounts line-by-line using a 
    smart noise filter to ignore OCR artifacts like 'Rs' being read as '2'.
    """
    # Structure: [Land_Est, Land_Inc, Dev_Est, Dev_Inc]
    amounts = [0.0, 0.0, 0.0, 0.0]
    
    try:
        images = convert_from_path(pdf_path, poppler_path=POPPLER_PATH)
        full_text = ""
        
        for img in images:
            open_cv_image = np.array(img)
            gray = cv2.cvtColor(open_cv_image, cv2.COLOR_BGR2GRAY)
            thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
            full_text += pytesseract.image_to_string(thresh) + "\n"
        
        # Process the extracted text line-by-line
        lines = full_text.split('\n')
        
        for line in lines:
            # Clean global symbols that confuse OCR
            line = line.replace('%', '')
            
            # Determine which row we are looking at
            is_land = re.search(r"Sub[\s-]*Total\s*(?:of\s*)?LAND\s*COST", line, re.IGNORECASE)
            is_dev = re.search(r"Sub[\s-]*Total\s*(?:of\s*)?(?:Dev|bev|deb)[a-zA-Z]*\s*Cost", line, re.IGNORECASE)
            
            if is_land or is_dev:
                # Find all contiguous blocks of numbers/commas/decimals
                raw_matches = re.findall(r'[\d,]+(?:\.\d+)?', line)
                valid_nums = []
                
                # FILTER LOGIC: Destroy the stray '2's
                for m in raw_matches:
                    clean_m = m.replace(',', '').strip()
                    try:
                        val = float(clean_m)
                        # Only keep the number if it has decimals (like 0.00), is exactly 0, or is a large number
                        # This perfectly drops stray single digits from misread 'Rs.' symbols
                        if '.' in clean_m or val == 0.0 or val > 99:
                            valid_nums.append(val)
                    except ValueError:
                        pass
                
                # The last two valid numbers left in the list are guaranteed to be Est and Inc
                if len(valid_nums) >= 2:
                    if is_land:
                        amounts[0] = valid_nums[-2]
                        amounts[1] = valid_nums[-1]
                    elif is_dev:
                        amounts[2] = valid_nums[-2]
                        amounts[3] = valid_nums[-1]

    except Exception as e:
        print(f"❌ OCR Extraction Error: {e}")
    finally:
        if os.path.exists(pdf_path):
            try:
                os.remove(pdf_path)
            except Exception:
                pass
            
    return amounts

def process_r3_top_folders():
    """Traverses folders, extracts Top-half R3 data, performs math, and checks matches."""
    for root, dirs, files in os.walk(ROOT_DIR):
        for file in files:
            if file.startswith("Form_R3_Top") and file.endswith((".xlsx", ".xls")) and not file.startswith("Compared_"):
                excel_path = os.path.join(root, file)
                print(f"\n" + "="*60)
                print(f"Processing R3 Top: {excel_path}")
                print("="*60)
                
                try:
                    df = pd.read_excel(excel_path)
                    
                    docs_col = [col for col in df.columns if 'docs' in col.lower()]
                    est_amt_col = [col for col in df.columns if 'estimated amount' in col.lower()]
                    curr_qtr_col = [col for col in df.columns if 'current quarter' in col.lower()]
                    
                    if not docs_col or not est_amt_col or not curr_qtr_col:
                        print("Skipping: Required columns missing from this excel sheet.")
                        continue
                        
                    docs_col_name = docs_col[0]
                    est_amt_col_name = est_amt_col[0]
                    curr_qtr_col_name = curr_qtr_col[0]
                    
                    pdf_url = df[docs_col_name].dropna().iloc[0] if not df[docs_col_name].dropna().empty else None
                    if not pdf_url:
                        print("Skipping: No PDF download link found.")
                        continue
                        
                    print(f"📥 Downloading PDF from Link...")
                    temp_pdf_path = download_pdf(pdf_url)
                    
                    if temp_pdf_path:
                        print("🔍 Extracting Blue-Boxed rows with Noise Filtering...")
                        extracted_vals = extract_r3_top_amounts_from_pdf(temp_pdf_path)
                        
                        land_est_pdf, land_inc_pdf = extracted_vals[0], extracted_vals[1]
                        dev_est_pdf, dev_inc_pdf = extracted_vals[2], extracted_vals[3]
                        
                        # Row C Math (Sum of A + B columns)
                        total_est_pdf = land_est_pdf + dev_est_pdf
                        total_inc_pdf = land_inc_pdf + dev_inc_pdf
                        
                        # Print extracted metrics directly to console
                        print(f"\n--- PDF EXTRACTION RESULTS ---")
                        print(f"Row (A) Land Cost Sub-Total        -> Estimated: {land_est_pdf:,.2f} | Incurred: {land_inc_pdf:,.2f}")
                        print(f"Row (B) Development Cost Sub-Total -> Estimated: {dev_est_pdf:,.2f} | Incurred: {dev_inc_pdf:,.2f}")
                        print(f"Row (C) Total project cost (A+B)   -> Estimated: {total_est_pdf:,.2f} | Incurred: {total_inc_pdf:,.2f}\n")
                        
                        pdf_extracted_est = [land_est_pdf, dev_est_pdf, total_est_pdf]
                        pdf_extracted_inc = [land_inc_pdf, dev_inc_pdf, total_inc_pdf]
                        
                        while len(pdf_extracted_est) < len(df):
                            pdf_extracted_est.append(0.0)
                            pdf_extracted_inc.append(0.0)
                            
                        pdf_extracted_est = pdf_extracted_est[:len(df)]
                        pdf_extracted_inc = pdf_extracted_inc[:len(df)]
                        
                        df['PDF_Extracted_Estimated'] = pdf_extracted_est
                        df['PDF_Extracted_Incurred(Current Qtr)'] = pdf_extracted_inc
                        df['Estimated_Match_Status'] = ""
                        df['Incurred_Match_Status'] = ""
                        
                        for index, row in df.iterrows():
                            excel_est = clean_amount(row[est_amt_col_name])
                            excel_inc = clean_amount(row[curr_qtr_col_name])
                            
                            pdf_est = row['PDF_Extracted_Estimated']
                            pdf_inc = row['PDF_Extracted_Incurred(Current Qtr)']
                            
                            df.at[index, 'Estimated_Match_Status'] = "Match" if excel_est == pdf_est else "Mismatch"
                            df.at[index, 'Incurred_Match_Status'] = "Match" if excel_inc == pdf_inc else "Mismatch"
                        
                        output_filename = f"Compared_{file}"
                        output_path = os.path.join(root, output_filename)
                        
                        try:
                            df.to_excel(output_path, index=False)
                            print(f"✅ Comparison saved to: {output_path}")
                        except PermissionError:
                            print(f"❌ [ACCESS DENIED]: Please close '{output_filename}' in Excel and run again!")
                            
                except Exception as e:
                    print(f"❌ Failed to process workbook {file}: {e}")

if __name__ == "__main__":
    print("🚀 Starting Form R3_Top Verification Run...")
    process_r3_top_folders()
    print("\n🏁 Form R3_Top Process Complete!")

🚀 Starting Form R3_Top Verification Run...

Processing R3 Top: C:\Users\Samarth Chaudhary\Downloads\RIDHI SIDHI COMMERCIAL HUB BLOCK A\sub\RIDHI SIDHI COMMERCIAL HUB BLOCK A\April to June, 2025\Form_R3_Top.xlsx
📥 Downloading PDF from Link...
🔍 Extracting Blue-Boxed rows with Noise Filtering...

--- PDF EXTRACTION RESULTS ---
Row (A) Land Cost Sub-Total        -> Estimated: 0.00 | Incurred: 0.00
Row (B) Development Cost Sub-Total -> Estimated: 0.00 | Incurred: 0.00
Row (C) Total project cost (A+B)   -> Estimated: 0.00 | Incurred: 0.00

✅ Comparison saved to: C:\Users\Samarth Chaudhary\Downloads\RIDHI SIDHI COMMERCIAL HUB BLOCK A\sub\RIDHI SIDHI COMMERCIAL HUB BLOCK A\April to June, 2025\Compared_Form_R3_Top.xlsx

Processing R3 Top: C:\Users\Samarth Chaudhary\Downloads\RIDHI SIDHI COMMERCIAL HUB BLOCK A\sub\RIDHI SIDHI COMMERCIAL HUB BLOCK A\January to March, 2026\Form_R3_Top.xlsx
📥 Downloading PDF from Link...
❌ Error downloading PDF from Not Found: Invalid URL 'Not Found': No scheme su

In [24]:
import os
import pandas as pd
import requests
import re
import tempfile
from pdf2image import convert_from_path
import pytesseract
import cv2
import numpy as np

# ==============================================================================
# CONFIGURATION PATHS
ROOT_DIR = r"C:\Users\Samarth Chaudhary\Downloads\RIDHI SIDHI COMMERCIAL HUB BLOCK A"
POPPLER_PATH = r"C:\Users\Samarth Chaudhary\Downloads\poppler\Poopler\Library\bin"
TESSERACT_PATH = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
# ==============================================================================

pytesseract.pytesseract.tesseract_cmd = TESSERACT_PATH

def download_pdf(url):
    """Downloads a PDF from a given URL to a temporary file safely."""
    try:
        response = requests.get(url, stream=True, timeout=30)
        response.raise_for_status()
        temp_pdf = tempfile.NamedTemporaryFile(delete=False, suffix=".pdf")
        with open(temp_pdf.name, 'wb') as f:
            f.write(response.content)
        return temp_pdf.name
    except Exception as e:
        print(f"❌ Error downloading PDF from {url}: {e}")
        return None

def clean_amount(val):
    """Converts Excel cell values into clean, standardized floats for comparison."""
    if pd.isna(val):
        return 0.0
    val_str = str(val).strip().upper()
    if val_str in ["", "NA", "N/A", "NONE", "NULL", "0", "-", ".00"]:
        return 0.0
    val_str = val_str.replace("RS.", "").replace("RS", "").replace(",", "").replace("%", "").strip()
    try:
        return float(val_str)
    except ValueError:
        return 0.0

def extract_r3_bottom_amounts_from_pdf(pdf_path):
    """
    Extracts numerical values for rows 5 to 9 using a dual-PSM layout strategy 
    and strict contextual phrase containment filters.
    """
    extracted_data = {5: 0.0, 6: 0.0, 7: 0.0, 8: 0.0, 9: 0.0}
    # Track whether the current value was populated using an explicit S.No prefix line
    has_prefixed_match = {5: False, 6: False, 7: False, 8: False, 9: False}
    
    try:
        images = convert_from_path(pdf_path, dpi=300, poppler_path=POPPLER_PATH)
        full_text = ""
        
        for img in images:
            open_cv_image = np.array(img)
            gray = cv2.cvtColor(open_cv_image, cv2.COLOR_RGB2GRAY)
            _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            
            # Dual-PSM execution prevents boxed bottom rows (Row 9) from being skipped
            text_psm6 = pytesseract.image_to_string(thresh, config='--psm 6')
            text_psm3 = pytesseract.image_to_string(thresh, config='--psm 3')
            
            full_text += text_psm6 + "\n" + text_psm3 + "\n"
        
        lines = full_text.split('\n')
        for line in lines:
            clean_line = line.replace('%', '').replace('₹', '').replace('Rs.', '').strip()
            if not clean_line:
                continue
                
            target_sno = None
            
            # --- STRENGTHENED CONTEXTUAL MATCHING RULES ---
            if re.search(r"Net\s*Amount\s*which\s*can\s*be\s*withdrawn", clean_line, re.IGNORECASE):
                target_sno = 9
            elif re.search(r"Less\s*:\s*Amount\s*withdrawn\s*till\s*date", clean_line, re.IGNORECASE) or \
                 (re.search(r"Less", clean_line, re.IGNORECASE) and re.search(r"till\s*date\s*of\s*this\s*certificate", clean_line, re.IGNORECASE)):
                target_sno = 8
            elif re.search(r"Amount\s*which\s*can\s*be\s*withdrawn\s*from\s*the\s*(?:RERA|separate)", clean_line, re.IGNORECASE):
                target_sno = 7
            elif re.search(r"Percentage", clean_line, re.IGNORECASE) and re.search(r"Land\s*Cost", clean_line, re.IGNORECASE) and re.search(r"Total", clean_line, re.IGNORECASE):
                target_sno = 5
            elif re.search(r"Percentage", clean_line, re.IGNORECASE) and re.search(r"(?:Construction|Development)", clean_line, re.IGNORECASE) and re.search(r"Total", clean_line, re.IGNORECASE):
                target_sno = 6
                
            if target_sno is not None:
                raw_matches = re.findall(r'[\d,]+(?:\.\d+)?', clean_line)
                valid_nums = []
                for m in raw_matches:
                    num_str = m.replace(',', '').strip()
                    if num_str and num_str != '.':
                        try:
                            valid_nums.append(float(num_str))
                        except ValueError:
                            pass
                
                if valid_nums:
                    extracted_val = valid_nums[-1]
                    
                    # Detect structural serial number prefixes (e.g., "7 |", "7 ", or starting with "7")
                    is_prefixed = bool(re.match(rf"^\s*{target_sno}\b", clean_line)) or \
                                  bool(re.match(rf"^\s*{target_sno}\s*\|", clean_line))
                    
                    # Protect established entries from being overwritten by un-prefixed later page content
                    if not has_prefixed_match[target_sno]:
                        extracted_data[target_sno] = extracted_val
                        if is_prefixed:
                            has_prefixed_match[target_sno] = True
                        print(f"   [OCR Matched Row {target_sno}]: \"{clean_line[:60]}...\" -> {extracted_val}")
                    elif is_prefixed:
                        # Update even if already matched if this line contains the explicit prefix tracking token
                        extracted_data[target_sno] = extracted_val
                        print(f"   [OCR Updated Row {target_sno} via Prefix]: \"{clean_line[:60]}...\" -> {extracted_val}")

    except Exception as e:
        print(f"❌ OCR Extraction Error: {e}")
    finally:
        if os.path.exists(pdf_path):
            try:
                os.remove(pdf_path)
            except Exception:
                pass
                
    return extracted_data

def process_r3_bottom_folders():
    """Traverses local workbook architecture and updates quarter validation records."""
    for root, dirs, files in os.walk(ROOT_DIR):
        for file in files:
            if file.startswith("Form_R3_Bottom") and file.endswith((".xlsx", ".xls")) and not file.startswith("Compared_"):
                excel_path = os.path.join(root, file)
                print(f"\n" + "="*75)
                print(f"Processing Sheet: {excel_path}")
                print("="*75)
                
                try:
                    df = pd.read_excel(excel_path)
                    
                    docs_col = [col for col in df.columns if 'docs' in col.lower()]
                    curr_qtr_col = [col for col in df.columns if 'current quarter' in col.lower() or 'in current' in col.lower()]
                    sno_col = [col for col in df.columns if 's.no' in col.lower() or 'sno' in col.lower()]
                    particulars_col = [col for col in df.columns if 'particulars' in col.lower()]
                    
                    if not docs_col or not curr_qtr_col:
                        print("Skipping: Required verification target columns missing.")
                        continue
                        
                    docs_col_name = docs_col[0]
                    curr_qtr_col_name = curr_qtr_col[0]
                    
                    pdf_url = df[docs_col_name].dropna().iloc[0] if not df[docs_col_name].dropna().empty else None
                    if not pdf_url:
                        print("Skipping: No active document path URL found.")
                        continue
                        
                    print(f"📥 Downloading PDF Target Asset...")
                    temp_pdf_path = download_pdf(pdf_url)
                    
                    if temp_pdf_path:
                        extracted_vals = extract_r3_bottom_amounts_from_pdf(temp_pdf_path)
                        
                        df['PDF_Extracted_Current_Qtr'] = 0.0
                        df['Current_Quarter_Match_Status'] = ""
                        
                        for index, row in df.iterrows():
                            detected_sno = None
                            
                            if sno_col:
                                s_val = str(row[sno_col[0]]).strip()
                                if s_val.isdigit():
                                    detected_sno = int(s_val)
                                    
                            if detected_sno is None and particulars_col:
                                text_look = str(row[particulars_col[0]]).lower()
                                if "land cost to the total estimated" in text_look:
                                    detected_sno = 5
                                elif "construction cost to the total estimated" in text_look:
                                    detected_sno = 6
                                elif "amount which can be withdrawn" in text_look and "net" not in text_look:
                                    detected_sno = 7
                                elif "less: amount withdrawn" in text_look:
                                    detected_sno = 8
                                elif "net amount which can be withdrawn" in text_look:
                                    detected_sno = 9
                            
                            pdf_amount = 0.0
                            if detected_sno in [5, 6, 7, 8, 9]:
                                pdf_amount = extracted_vals.get(detected_sno, 0.0)
                                
                            df.at[index, 'PDF_Extracted_Current_Qtr'] = pdf_amount
                            
                            excel_amount = clean_amount(row[curr_qtr_col_name])
                            if detected_sno in [5, 6, 7, 8, 9]:
                                is_match = abs(excel_amount - pdf_amount) < 0.1
                                df.at[index, 'Current_Quarter_Match_Status'] = "Match" if is_match else "Mismatch"
                            else:
                                df.at[index, 'Current_Quarter_Match_Status'] = "Ignored (Rows 2-4)"
                        
                        output_filename = f"Compared_{file}"
                        output_path = os.path.join(root, output_filename)
                        
                        try:
                            df.to_excel(output_path, index=False)
                            print(f"✅ Comparison report saved: {output_path}")
                        except PermissionError:
                            print(f"❌ [FILE LOCKED]: Close '{output_filename}' in Excel and retry.")
                            
                except Exception as e:
                    print(f"❌ Failed to execute spreadsheet processing on {file}: {e}")

if __name__ == "__main__":
    print("🚀 Initializing Form R3 Multi-Document Alignment Audit...")
    process_r3_bottom_folders()
    print("\n🏁 Validation sweep completed successfully!")

🚀 Initializing Form R3 Multi-Document Alignment Audit...

Processing Sheet: C:\Users\Samarth Chaudhary\Downloads\RIDHI SIDHI COMMERCIAL HUB BLOCK A\sub\RIDHI SIDHI COMMERCIAL HUB BLOCK A\April to June, 2025\Form_R3_Bottom.xlsx
📥 Downloading PDF Target Asset...
   [OCR Matched Row 5]: "5 Percentage of the Cost incurred on Land Cost to the Total ..." -> 90.0
✅ Comparison report saved: C:\Users\Samarth Chaudhary\Downloads\RIDHI SIDHI COMMERCIAL HUB BLOCK A\sub\RIDHI SIDHI COMMERCIAL HUB BLOCK A\April to June, 2025\Compared_Form_R3_Bottom.xlsx

Processing Sheet: C:\Users\Samarth Chaudhary\Downloads\RIDHI SIDHI COMMERCIAL HUB BLOCK A\sub\RIDHI SIDHI COMMERCIAL HUB BLOCK A\July to September, 2025\Form_R3_Bottom.xlsx
📥 Downloading PDF Target Asset...
   [OCR Matched Row 6]: "Percentage of the Cost incurred on Construction Cost to the ..." -> 0.0
   [OCR Matched Row 9]: "Net Amount which can be withdrawn from the Separate Bank Acc..." -> 22202245.0
   [OCR Matched Row 5]: "5 Percentage of the 